# Import pakcages

In [1]:
import json

# Custom functions

In [2]:
from tools.obtain_gene_seq import gene_name_to_id, gene_uid_to_gene_summary, get_strand_from_gene_efetch, determine_TSS_from_gene_summary, retrieve_seq, gene_name_to_download_seq

# Function testing by each step

## Obtain gene_name → gene_uid

In [3]:
gene_name = 'EGFR'

gene_uid = gene_name_to_id(gene_name) # gene_uid is a str
print(gene_uid) # gene_uid can be verified by NIH gene database. For example, for EGFR, it has https://www.ncbi.nlm.nih.gov/gene/?term=EGFR showing the Gene ID as 1956

1956


## Obtain gene_uid → gene_summary

In [4]:
gene_summary = gene_uid_to_gene_summary(gene_uid)

print(json.dumps(gene_summary, indent =4))

>>>> Gene summary found for epidermal growth factor receptor
{
    "uid": "1956",
    "name": "EGFR",
    "description": "epidermal growth factor receptor",
    "status": "",
    "currentid": "",
    "chromosome": "7",
    "geneticsource": "genomic",
    "maplocation": "7p11.2",
    "otheraliases": "ERBB, ERBB1, ERRP, HER1, NISBD2, NNCIS, PIG61, mENA",
    "otherdesignations": "epidermal growth factor receptor|EGFR vIII|avian erythroblastic leukemia viral (v-erb-b) oncogene homolog|cell growth inhibiting protein 40|cell proliferation-inducing protein 61|epidermal growth factor receptor tyrosine kinase domain|erb-b2 receptor tyrosine kinase 1|proto-oncogene c-ErbB-1|receptor tyrosine-protein kinase erbB-1",
    "nomenclaturesymbol": "EGFR",
    "nomenclaturename": "epidermal growth factor receptor",
    "nomenclaturestatus": "Official",
    "mim": [
        "131550"
    ],
    "genomicinfo": [
        {
            "chrloc": "7",
            "chraccver": "NC_000007.14",
            "chr

## Obtain strand information

In [5]:
strand = get_strand_from_gene_efetch(gene_uid)
print(strand) # This can be verified on the specific NIH gene page under the Genomic context tab's Location field. For example, for EGFR, it has https://www.ncbi.nlm.nih.gov/gene/1956 and show that all the Location are in the format like NC_000007.14 (55019017..55211628), which means the gene is on the plus strand. If a gene is on the minus strand, it will be in the format like NC_000007.14 (5527148..5530601, complement)

plus


## Obtain TTS

In [6]:
chromosome, strand, tss = determine_TSS_from_gene_summary(gene_summary)

print(chromosome, strand, tss)

NC_000007.14 plus 55019017


## Retrieve sequence

In [7]:
seq = retrieve_seq(
    gene_name,
    gene_uid,
    chromosome,
    strand,
    tss,
    seq_type = 'promoter',
    upstream_bp = 100,
    downstream_bp = 100
)

print(seq)

>NIH gene ID: 1956 EGFR | NC_000007.14:55018917-55019116 Homo sapiens chromosome 7, GRCh38.p14 Primary Assembly | Strand: plus | Type: promoter | TSS (on chromosome): 55019017 | TSS (on sequence): 100
CCTCGCATTCTCCTCCTCCTCTGCTCCTCCCGATCCCTCCTCCGCCGCCTGGTCCCTCCTCCTCCCGCCCTGCCTCCCCGCGCCTCGGCCCGCGCGAGCTAGACGTCCGGGCAGCCCCCGGCGCAGCGCGGCCGCAGCAGCCTCCGCCCCCCGCACGGTGTGAGCGCCCGACGCGGCCGAGGCGGCCGGAGTCCCGAGCT


Reference sequence for EGFR of 100 bp upstream and 100 bp dowstream of the gene's TSS is:
>NM_005228 EGFR | Homo sapiens chromosome 7, GRCh38.p14 Primary Assembly NC_000007.14 | Strand: plus | Type: promoter | TSS (on chromosome): 55019017 | TSS (on sequence): 100
CCTCGCATTCTCCTCCTCCTCTGCTCCTCCCGATCCCTCCTCCGCCGCCTGGTCCCTCCTCCTCCCGCCCTGCCTCCCCGCGCCTCGGCCCGCGCGAGCTAGACGTCCGGGCAGCCCCCGGCGCAGCGCGGCCGCAGCAGCCTCCGCCCCCCGCACGGTGTGAGCGCCCGACGCGGCCGAGGCGGCCGGAGTCCCGAGCT

# Test the whole wrapper function
You need to test both a gene on the plus and another on the minus strand

You can use the NIH blastn tool to compare whether the sequences match the references: https://blast.ncbi.nlm.nih.gov/Blast.cgi?PAGE=MegaBlast&PROGRAM=blastn&BLAST_PROGRAMS=megaBlast&PAGE_TYPE=BlastSearch&BLAST_SPEC=blast2seq&DATABASE=n/a&QUERY=&SUBJECTS=

## Gene on minus strand
ACTB as an example
* NIH gene page: https://www.ncbi.nlm.nih.gov/gene/60

In [8]:
gene_uid = gene_name_to_id('ACTB')
strand = get_strand_from_gene_efetch(gene_uid)
print(strand)

minus


In [9]:
seq = gene_name_to_download_seq(
    gene_name='ACTB',
    seq_type = 'promoter',
    upstream_bp = 5000,
    downstream_bp = 2000,
    save_file = True
)

print(seq)

>>>> Gene summary found for actin beta
>NIH gene ID: 60 ACTB | NC_000007.14:c5535601-5528602 Homo sapiens chromosome 7, GRCh38.p14 Primary Assembly | Strand: minus | Type: promoter | TSS (on chromosome): 5530601 | TSS (on sequence): 5000
CGGGAGGCAGAGGTTGCAGTGAGCTGAGATTGTGCCACTGCAGTCCAGCCTGGGCGACAGAGCAAGACTCCATCTCAAAACAACAACAACTAAAAACGAACAACAACAACAACAAAAAACTGAGGCCAGGTGCAGTGGCTCACACCTCTAATCCCAGCAATTTGGGAGGCTGAGGTGAGAGGATCACTTCAGCTCAGCAGTTCGAGACCAGCCCAGGCAACACAGGGAGACAGACCCTGTTATATTGCAGAGAGACCCCATCTCCACAAAATATAAAAATATTAGTCAGATGTGGTGGCATCCCTGCCGTCCCAGCTACTCAGGAGGCTGAGACAGGAGGATCGCTTGAGCCCAGGAGGTCAAGGCTGCAGTGAGCTGTGATCATGCCACTGCACTCCAGCCTGGGCCACAGAGCTAGACCCTGTCTCTAAAATTTTTAGAGACCTTATCTCTAAAAATAAATTAAATAAATAAACCGGGAGCACCTACTTTTTCTTTTTCTTTTACTTTTTTTTTTTTTTTTTGGAGACAGGGTCTCTATCACCCAGGCTGGAGTGCGGTGGCATGATCTTGGTTCACTGCAGCCTCGACCCCTCAGGCTCAGGCAGTTCTCCCACCTTAGCCTCCCCAGTAGCTGGGACTACAGGCACATGCCACCATGCCCGGCTAATTTTGTCTTTTTTTTTTTTTTTGGTAGAGACAGGGTCTCACCATGCTCCTCAAAACTCCTGGACTCGAGAGATCCTCCTGCCTCGGCCTCCC

Reference sequence for ACTB of 5000 bp upstream and 2000 bp dowstream of the gene's TSS is:
>NM_001101 ACTB | Homo sapiens chromosome 7, GRCh38.p14 Primary Assembly NC_000007.14 | Strand: minus | Type: promoter | TSS (on chromosome): 5530601 | TSS (on sequence): 5000
CGGGAGGCAGAGGTTGCAGTGAGCTGAGATTGTGCCACTGCAGTCCAGCCTGGGCGACAGAGCAAGACTCCATCTCAAAACAACAACAACTAAAAACGAACAACAACAACAACAAAAAACTGAGGCCAGGTGCAGTGGCTCACACCTCTAATCCCAGCAATTTGGGAGGCTGAGGTGAGAGGATCACTTCAGCTCAGCAGTTCGAGACCAGCCCAGGCAACACAGGGAGACAGACCCTGTTATATTGCAGAGAGACCCCATCTCCACAAAATATAAAAATATTAGTCAGATGTGGTGGCATCCCTGCCGTCCCAGCTACTCAGGAGGCTGAGACAGGAGGATCGCTTGAGCCCAGGAGGTCAAGGCTGCAGTGAGCTGTGATCATGCCACTGCACTCCAGCCTGGGCCACAGAGCTAGACCCTGTCTCTAAAATTTTTAGAGACCTTATCTCTAAAAATAAATTAAATAAATAAACCGGGAGCACCTACTTTTTCTTTTTCTTTTACTTTTTTTTTTTTTTTTTGGAGACAGGGTCTCTATCACCCAGGCTGGAGTGCGGTGGCATGATCTTGGTTCACTGCAGCCTCGACCCCTCAGGCTCAGGCAGTTCTCCCACCTTAGCCTCCCCAGTAGCTGGGACTACAGGCACATGCCACCATGCCCGGCTAATTTTGTCTTTTTTTTTTTTTTTGGTAGAGACAGGGTCTCACCATGCTCCTCAAAACTCCTGGACTCGAGAGATCCTCCTGCCTCGGCCTCCCAAAATGCTGGGATTACGGTGTGAGCCGCTGTGCCCGGCTATTTTATTTTTAAATGAATAAAAGCTGGAGCACCCAACTTTTTTGTTGTTGTTTTTCTGAGACAGAGTTTTGCTCGTCACCCAGGCTGGAGTGCAGCGGCGCAATCTCGGCCCACTGCAACCTCTGCCTCCCGGGTTCAAGCGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGATTACAGGCACGCGCCACTATGCCCAGCTAATATTTTGTATTTTTAGTAGAGACAAGGTTTCATCATGTTTGCCAGGCTGATCTTGAATTTTTGACCTCAGGTGATCCACCCGCCTTGACCTCCCAAAGTGCAGGGATTACAGGCGTGAGCCACCACGCCCAGCCTGGAGCACCTAATTTTTAAATTTAATTTTTCTTTTCTTTTTTTTTTTTTTTTGAGACAGGGTCTTCCTCTGTCATCCAAGCTGGAATGCAGTGACTTGATCATAGCTCACTATAACCTTGACCTTCCACGCTGAAGCAATGCTCCTGCCCCAGCCTCCCAATTAGCTGGGAGTACAGGCACGTGCCACAACACCCAGCTGATTTTGTAGAGATAGGATCTCCCGTGTTGCCCAAGCTTGTCTCCAACTCCTGGGCTCGAGCGATCTTCCCTCCTCGGCCTCCCAAAATGCTGGGATTACAGGTACAAGCCATCACACCCCAGTGGGAGAGACCCCACTTGCTGCCACGTGACCATGGGCTGATGGTGTCCTCTCTGGGCCTCAGGTGATAAATTCTGAGGAGGGAGGTAGGGCCCAGGAATTCCGACTTTCAAACAGCCTTTGGACTCAGGCCTTGGGGACATTCCAGGGGACCCTTTACCAGGGAGGGGCAGTGTGAGGCAGGAGGCAGGGCTGCCGCCTCAGGGACCCTGAGGCTAGAGTCCTTCCCACCGCATTTGGGGACTTGAGCACTTCACCAGTCCTGACTCAGTTTCCTAGTCTGTGAATGGGGGTGTGTCTCCTGGAATGTTGCCCCCCCTCATTCATTCATTCTCCTCTCACTGAGCACCTACCGTGTTCAGGGTCCCTGTCCTCCTAGAGCTAACTCAAGTCTTGGAGGGAAGCTCTGTCTCAGTTCCTTCCTCTACAAAATGGAGCTTGAGGACGGGCACGGTGGCTCATGCCTGTAATCCCAGGACTTTGGGAGGCTGAGGTGGGAGGATCGCTTGAGCCCCCCGGAGCTGGAGACCAGCCTGGGCAACATGGTGAGAAGCCATACCTCTACAGATAATTTAAAAATTAGCCAGGCTTGGAAGTGTGTGCCTATAGTCCCAGCTACTCGTGAAGGCTTTGAGCCCAGGAGGTCGAGGCTGCAGTGAGCTGTGATCGCACTCCAGCCTGGGCATCACAGCAAGACCTTGTATCAAAAATAGTAATAATAATAAAAAGGAGGTTGGATTCCCTCCTGGCAGGATAGGGAGGGCGCTGCAGTGCCCAGGGCAAGGTGGCTGGGTGGTTGTTTTGCGGGAGGGCCAAGGAGTGGTCCCTGGGTCTGCGCTGTAAGAGTTGGTTGCCTGGAGGCCTTGCAGGGTGGGGGTGTCACCAAAGCAAAGGCTTGGAGGGGAAAAAACAAAAGTCCCAGCCAGGCAGGTGGAGTCCCCTTGGCTGAGTTCAACCGAGGGTTCTCCGGGGGCTGCGTGCGTGCCCCAGTGACAGCTCCGAAAGCTCCCTTACAGGGCAAAGTTCCCAAGCACAGAAGAGAACCTGTTCACTTCTCCCCTGCTCGGCCCGCCCCCTGGCCAGGCACCTCTACTTCCTCTTTTCCTGCTCCGCTGCTTGCTTTCTCTCTTCAGCTCCTCCCTGCCCCTCACCCCAGGCTGCTCGGCCACCTCCAACCTGCCACCTGAGGACACCCAGGCAGTCACTCATTCAACAGCGAGGAGCCCTGGGGTGGGTGTAGTGGGAAGGAGTGGGGGTGACGGAGACCCTGGGAGGGCTCGCAGCCTGGTGGCTGAGGCCCAGTTCTAAATGCCAGCTGCAAGCCTTGGTCTGAGGTAGGGAGGAAGGCGTGGCTGCAGAGGCTAAAACGCTTCCCCAAAGAGGGGCTTTCTGGGATGGGACTTGAAGGGTGCATAGGAGAGCACTAGGAAGTGGCCGCTGCAGACAGAGGGAACCACAAGCCAGGAGGACAGGCCAGGAATGCTGCAGCCCGGGGCGGGGTGGGGCTGGAGCTCCTGTCTCTTGGCCAGCTGAATGGAGGCCCAGTGGCAACACAGGTCCTGCCTGGGGATCAGGTCTGCTCTGCACCCCACCTTGCTGCCTGGAGCCGCCCACCTGACAACCTCTCATCCCTGCTCTGCAGATCCGGTCCCATCCCCACTGCCCACCCCACCCCCCCAGCACTCCACCCAGTTCAACGTTCCACGAACCCCCAGAACCAGCCCTCATCAACAGGCAGCAAGAAGGGCCCCCCGCCCATCGCCCCACAACGCCAGCCGGGTGAACGTTGGCAGGTCCTGAGGCAGCTGGCAAGACGCCTGCAGCTGAAAGATACAAGGCCAGGGACAGGACAGTCCCATCCCCAGGAGGCAGGGAGTATACAGGCTGGGGAAGTTTGCCCTTGCGTGGGGTGGTGATGGAGGAGGCTCAGCAAGTCTTCTGGACTGTGAACCTGTGTCTGCCACTGTGTGCTGGGTGGTGGTCATCTTTCCCACCAGGCTGTGGCCTCTGCAACCTTCAAGGGAGGAGCAGGTCCCATTGGCTGAGCACAGCCTTGTACCGTGAACTGGAACAAGCAGCCTCCTTCCTGGCCACAGGTTCCATGTCCTTATATGGACTCATCTTTGCCTATTGCGACACACACTCAGTGAACACCTACTACGCGCTGCAAAGAGCCCCGCAGGCCTGAGGTGCCCCCACCTCACCACTCTTCCTATTTTTGTGTAAAAATCCAGCTTCTTGTCACCACCTCCAAGGAGGGGGAGGAGGAGGAAGGCAGGTTCCTCTAGGCTGAGCCGAATGCCCCTCTGTGGTCCCACGCCACTGATCGCTGCATGCCCACCACCTGGGTACACACAGTCTGTGATTCCCGGAGCAGAACGGACCCTGCCCACCCGGTCTTGTGTGCTACTCAGTGGACAGACCCAAGGCAAGAAAGGGTGACAAGGACAGGGTCTTCCCAGGCTGGCTTTGAGTTCCTAGCACCGCCCCGCCCCCAATCCTCTGTGGCACATGGAGTCTTGGTCCCCAGAGTCCCCCAGCGGCCTCCAGATGGTCTGGGAGGGCAGTTCAGCTGTGGCTGCGCATAGCAGACATACAACGGACGGTGGGCCCAGACCCAGGCTGTGTAGACCCAGCCCCCCCGCCCCGCAGTGCCTAGGTCACCCACTAACGCCCCAGGCCTTGTCTTGGCTGGGCGTGACTGTTACCCTCAAAAGCAGGCAGCTCCAGGGTAAAAGGTGCCCTGCCCTGTAGAGCCCACCTTCCTTCCCAGGGCTGCGGCTGGGTAGGTTTGTAGCCTTCATCACGGGCCACCTCCAGCCACTGGACCGCTGGCCCCTGCCCTGTCCTGGGGAGTGTGGTCCTGCGACTTCTAAGTGGCCGCAAGCCACCTGACTCCCCCAACACCACACTCTACCTCTCAAGCCCAGGTCTCTCCCTAGTGACCCACCCAGCACATTTAGCTAGCTGAGCCCCACAGCCAGAGGTCCTCAGGCCCTGCTTTCAGGGCAGTTGCTCTGAAGTCGGCAAGGGGGAGTGACTGCCTGGCCACTCCATGCCCTCCAAGAGCTCCTTCTGCAGGAGCGTACAGAACCCAGGGCCCTGGCACCCGTGCAGACCCTGGCCCACCCCACCTGGGCGCTCAGTGCCCAAGAGATGTCCACACCTAGGATGTCCCGCGGTGGGTGGGGGGCCCGAGAGACGGGCAGGCCGGGGGCAGGCCTGGCCATGCGGGGCCGAACCGGGCACTGCCCAGCGTGGGGCGCGGGGGCCACGGCGCGCGCCCCCAGCCCCCGGGCCCAGCACCCCAAGGCGGCCAACGCCAAAACTCTCCCTCCTCCTCTTCCTCAATCTCGCTCTCGCTCTTTTTTTTTTTCGCAAAAGGAGGGGAGAGGGGGTAAAAAAATGCTGCACTGTGCGGCGAAGCCGGTGAGTGAGCGGCGCGGGGCCAATCAGCGTGCGCCGTTCCGAAAGTTGCCTTTTATGGCTCGAGCGGCCGCGGCGGCGCCCTATAAAACCCAGCGGCGCGACGCGCCACCACCGCCGAGACCGCGTCCGCCCCGCGAGCACAGAGCCTCGCCTTTGCCGATCCGCCGCCCGTCCACACCCGCCGCCAGGTAAGCCCGGCCAGCCGACCGGGGCAGGCGGCTCACGGCCCGGCCGCAGGCGGCCGCGGCCCCTTCGCCCGTGCAGAGCCGCCGTCTGGGCCGCAGCGGGGGGCGCATGGGGGGGGAACCGGACCGCCGTGGGGGGCGCGGGAGAAGCCCCTGGGCCTCCGGAGATGGGGGACACCCCACGCCAGTTCGGAGGCGCGAGGCCGCGCTCGGGAGGCGCGCTCCGGGGGTGCCGCTCTCGGGGCGGGGGCAACCGGCGGGGTCTTTGTCTGAGCCGGGCTCTTGCCAATGGGGATCGCAGGGTGGGCGCGGCGGAGCCCCCGCCAGGCCCGGTGGGGGCTGGGGCGCCATTGCGCGTGCGCGCTGGTCCTTTGGGCGCTAACTGCGTGCGCGCTGGGAATTGGCGCTAATTGCGCGTGCGCGCTGGGACTCAAGGCGCTAACTGCGCGTGCGTTCTGGGGCCCGGGGTGCCGCGGCCTGGGCTGGGGCGAAGGCGGGCTCGGCCGGAAGGGGTGGGGTCGCCGCGGCTCCCGGGCGCTTGCGCGCACTTCCTGCCCGAGCCGCTGGCCGCCCGAGGGTGTGGCCGCTGCGTGCGCGCGCGCCGACCCGGCGCTGTTTGAACCGGGCGGAGGCGGGGCTGGCGCCCGGTTGGGAGGGGGTTGGGGCCTGGCTTCCTGCCGCGCGCCGCGGGGACGCCTCCGACCAGTGTTTGCCTTTTATGGTAATAACGCGGCCGGCCCGGCTTCCTTTGTCCCCAATCTGGGCGCGCGCCGGCGCCCCCTGGCGGCCTAAGGACTCGGCGCGCCGGAAGTGGCCAGGGCGGGGGCGACCTCGGCTCACAGCGCGCCCGGCTATTCTCGCAGCTCACCATGGATGATGATATCGCCGCGCTCGTCGTCGACAACGGCTCCGGCATGTGCAAGGCCGGCTTCGCGGGCGACGATGCCCCCCGGGCCGTCTTCCCCTCCATCGTGGGGCGCCCCAGGCACCAGGTAGGGGAGCTGGCTGGGTGGGGCAGCCCCGGGAGCGGGCGGGAGGCAAGGGCGCTTTCTCTGCACAGGAGCCTCCCGGTTTCCGGGGTGGGGGCTGCGCCCGTGCTCAGGGCTTCTTGTCCTTTCCTTCCCAGGGCGTGATGGTGGGCATGGGTCAGAAGGATTCCTATGTGGGCGACGAGGCCCAGAGCAAGAGAGGCATCCTCACCCTGAAGTACCCCATCGAGCACGGCATCGTCACCAACTGGGACGACATGGAGAAAATCTGGCACCACACCTTCTACAATGAGCTGCGTGTGGCTCCCGAGGAGCACCCCGTGCTGCTGACCGAGGCCCCCCTGAACCCCAAGGCCAACCGCGAGAAGATGACCCAGGTGAGTGGCCCGCTACCTCTTCTGGTGGCCGCCTCCCTCCTTCCTGGCCTCCCGGAGCTGCGCCCTTTCTCACTGGTTCTCTCTTCTGCCGTTTTCCGTAGGACTCTCTTCTCTGACCTGAGTCTCCTTTGGAACTCTGCAGGTTCTATTTGCTTTTTCCCAGATGAGCTCTTTTTCTGGTGTTTGTCTCTCTGACTAGGTGTCTAAGACAGTGTTGTGGGTGTAGGTACTAACACTGGCTCGTGTGACAAGGCCATGAGGCTGGTGTAAAGCGGCCTTGGAGTGTGTATTAAGTAGGTGCACAGTAGGTCTGAACAGACTCCCCATCCCAAGACCCCAGCACACTTAGCCGTGTTCTTTGCACTTTCTGCATGTCCCCCGTCTGGCCTGGCTGTCCCCAGTGGCTTCCCCAGTGTGACATGGTGTATCTCTGCCTTACAGATCATGTTTGAGACCTTCAACACCCCAGCCATGTACGTTGCTATCCAGGCTGTGCTATCCCTGTACGCCTCTGGCCGTACCACTGGCATCGTGATGGACTCCGGTGACGGGGTCACCC

## Gene on plus strand
PTEN as an example
* NIH gene page: https://www.ncbi.nlm.nih.gov/gene/5728

In [10]:
gene_uid = gene_name_to_id('PTEN')
strand = get_strand_from_gene_efetch(gene_uid)
print(strand)

plus


In [11]:
seq = gene_name_to_download_seq(
    gene_name='PTEN',
    seq_type = 'promoter',
    upstream_bp = 5000,
    downstream_bp = 2000,
    save_file = True
)

print(seq)

>>>> Gene summary found for phosphatase and tensin homolog
>NIH gene ID: 5728 PTEN | NC_000010.11:87858625-87865624 Homo sapiens chromosome 10, GRCh38.p14 Primary Assembly | Strand: plus | Type: promoter | TSS (on chromosome): 87863625 | TSS (on sequence): 5000
AGACTCCGTCTCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCTGCCTACTCTATGAAGCACCTGGTAGGTGTCCCAATATTTTCTGGTATATGGGAATTACCTTTTTCTATTATTGGAAGGTCTAAAAAGCAAACAATGTGCTCCATATGGCTAGAGTTAGTATTAAGGGGACCAAGTAGAATAATTGGTAGCTAGTGTTAGCCTGGAGACTAACATAGTCAAAGCTGTCTTGTCATGTTATGTCTTTTTTATTTGTATTGCTCCACGGTCCAAGACAAAATTTCTGAACGCCACTTGGACACAGTGAGTACCTGGTCTGCTCTATTGTTCTCAGGAGCAACCAACTCAACCCTTATTTCTCTGAGAATGATGATTTCATACAGCACATCTCTCTACCAAGATGTGAAAGATGACACCATGGCATCTGAAATAGCTTCAGGAGAGATTTGGGACATGGGAAGCTTGTAGACAATAATGGAAAATTCTCTTTTAGAATATAGTTACTTGTATGACCCACAGTAGCGCCTTTTGGAGAATGTCTTAAAATTATCTTTATTGAAAGAACAATAATGTTTGTTATTAGGATAAATGAAATAAGGGGAAAACCTCAGACCCTTGGAACAATGGGTTTACATTCAATCCAATGATTATTATGTTTTTATATTCTGTATTATTTAGAAAACAGTAGTTAAACAGACAGAAATACAGAAGATTGTTTAAAAATTAAAGCT

Reference sequence for PTEN of 5000 bp upstream and 2000 bp dowstream of the gene's TSS is:
>NM_000314 PTEN | Homo sapiens chromosome 10, GRCh38.p14 Primary Assembly NC_000010.11 | Strand: plus | Type: promoter | TSS (on chromosome): 87863625 | TSS (on sequence): 5000
AGACTCCGTCTCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCTGCCTACTCTATGAAGCACCTGGTAGGTGTCCCAATATTTTCTGGTATATGGGAATTACCTTTTTCTATTATTGGAAGGTCTAAAAAGCAAACAATGTGCTCCATATGGCTAGAGTTAGTATTAAGGGGACCAAGTAGAATAATTGGTAGCTAGTGTTAGCCTGGAGACTAACATAGTCAAAGCTGTCTTGTCATGTTATGTCTTTTTTATTTGTATTGCTCCACGGTCCAAGACAAAATTTCTGAACGCCACTTGGACACAGTGAGTACCTGGTCTGCTCTATTGTTCTCAGGAGCAACCAACTCAACCCTTATTTCTCTGAGAATGATGATTTCATACAGCACATCTCTCTACCAAGATGTGAAAGATGACACCATGGCATCTGAAATAGCTTCAGGAGAGATTTGGGACATGGGAAGCTTGTAGACAATAATGGAAAATTCTCTTTTAGAATATAGTTACTTGTATGACCCACAGTAGCGCCTTTTGGAGAATGTCTTAAAATTATCTTTATTGAAAGAACAATAATGTTTGTTATTAGGATAAATGAAATAAGGGGAAAACCTCAGACCCTTGGAACAATGGGTTTACATTCAATCCAATGATTATTATGTTTTTATATTCTGTATTATTTAGAAAACAGTAGTTAAACAGACAGAAATACAGAAGATTGTTTAAAAATTAAAGCTATTGAGTTAGATCCCTTTTTGAAAGGTCAGCGTATGGGAGATGAGAAAGGCACTATAGAGATCAGAGTGTTTACACAAAAGACATCTTAGCAGATGACCTACAAAGAGCACATCAAGTATTTATATCATCCACATCAAGTTGCTGGTCATTTCGCTTATCAAAGAAAATAAGAAAGAAATTTTCTTTCGACATTATTTTGGTGTAGTAACAATAGAGTTTTGGAATCAGCTGTTAGAAGCGATAATTAAAGCTAGTTTACCATGCTTACTAATCAATCTACATAGTCACCCTGAAGCTTTATATAATTGTCCCTTTGTTACAAGCTTCTACTTCTCCCTATGGTATTCTGGTTCTGAATCCAGACAGGTAAAGAACTAAGTATGGCCGGGCGCGGTGGCTCACGCCTGTAATCCCAGCACTTTGGGAGGCCGTGGCGGGCAGATCACGAGGTCAGGAGATCGAGACCATCCTGACCAACATGGTGAAACCCTGTCTTTACTAAACTACAAAAAATTAGCCCGGCATGATGGTGTGCGCCTGTAGTCCCAGCTACTCGGGAGGCTGAGGCAGGGGAATTGCTTGAACCTGGGAGGCAGAGGTTGCAGTGAACCGAGATCATGCCACTGCACTCCAGCCTGTGAGACTCCGTCTCAAAAAAAAAAAAAAAAAAAGAAAAAAAAAAAAAGAACTGAAGTGCTTGGAGACCAGTGTATTGTAATATCTATACAGCCAAGCAGGTATTTTCAACTGAACTTTAAAAAAGTACTATTTTTGAAACTTTCTTTAGTTTAGATAGGGCCAGCCATTTGGGTAGAAAGGAAAAGAAAAACAAAAAATTGGAAAAGGATGCATCAGTATTTCTTTCAGAAACAACTTAACACTAGGAACAGAAGGAGGATTTTATTCACCAACTCCTAATAACCACCTGATGTTGCTGTTGGGCTGGCCAAATATCTGCCAACATTGATGATCCTTTCATTCACCACTAGAGAGAAGTAGAGGGACCTGATGTTTAGAGAAGCAGTAAAACCCATTTCCCATTTTCTTTCTTCAGCCGGATGGAAGTACTTGGAGGCTGGTCTCATATTCTAGCACTTTAGTGCTGGTGATGCAGATTATCTCTGCTCCAGAGTAACTCTGTGCCATGGAATCCAAGAGGATTCTGTCGCACCAGGACAGGAATGCACCCTTGTTTCATTTGCTTCCATGAAAGCAGATTTCAGAGATTCTCAGATGCTATGAGACTAATTGTCACTTAGGGTAGGCTCCAGGGGAGGGACAGAGAAGGGCAGTGCTAATAACAGGGACTCTGGAGACAAGATCAGCAATCTAAGAAACAAATGTGAGACATTTTACCCCTCCCTAATCTACTCCCCTTTTAAAAGTTCATCTTCATTTTCCCTTTTTCTACCACGCAGTAATTGATAATTAAACACAGCTCTGGTTTTTACCATGCAATCTAGTAGAACACCTAGACATCAGGAACTCCTTATCTTTGGTTATTTATACAATAATGAAATCTGTAGTTCACGCTAGGCTGTGTGGACAGTATCTGCATCTCTGGAAATCAACTGGAGGCCTTTCTCTTACTCTTTTCCGGTTGCTTTCCATTTCCTTTTCCACTGTGGTCCTTGCCTTTCCCAAGCTGACCAGAACTATGCACAAAACCAGTCTTCAATAAATATCAACACTGAAGCTTCAGCAGGTTGTTAACTCCTGCACAGCTATCTTAAATGCTGTATATAAGTGCAAAGTTTTATGGAGCCTCAGAGGTCTCTAAGAACACTGTAAATATCCCAGGTTTAATTAGTAGTCCCGGAGTTAGGTAATGGCCTGTGGCTTTCTGCGCTAGGTCTCTTGAGGTTTCTAGTCTTCAGAGGTTGCCTGCAACTATGATGTGCACATATACAGTACTCTTTCTCAAATACAGCATAAACCCTCTTTAGACTTTGCTAGGCACTTACAATTATTGAACACACATGCAGATTGATTCTCATTCTCTCAAAGTTTGAAAAACAACTCAGTGCTTCAACCTAGGTCTCGTTAGATATTTTTTGACTCAAATTGTCGTCTGTAGTTCTACTTCCTAAGGGAAATGAAAAAACAATAAATTCCCAGACTGGTGTTGATGCTCATTCTCTTTAAGCGGGTCGACTACTTGCTTTGTAGATCCTTGACGGGTGGGGTGCGGGGTAGGAGTGCGATCCAACTCTCAGCATTTCCGAATCAGCTCTCTCACGGTGACAGGTCAGCCCAATCGGGGCTGTAAACAGACTTGACAGGTTTGTTCTGGGCTGACGGCCATTGACTAGGTTCTCAGACCAGATAAGTCACTTGGCTGAGTCCACAGTAGGTGGGGCGCGCTCACCAGCTCAGGGGTAGTGACTGGACGTTTGTTGCAACATCGGAGAATGCACGCTCTGGGCTGCAGCAGGAGATACCCTCAAGCACAGAACCAAAAGGGTTCACCCTAAGCGGCAGGGCATCAGCGATGGAGAGGCCCGAGAGCCCTAGCGCCCAGCTCCTTTTCCCACGTTTGGGAAGGCGCAGAATAGGTCGATGTAGAGCAAGGAGTGAGTCTCAGGTCTCAGTCCTTTGGCTTGCTCTTAGGGTAGCAGGCGAGGAGTGGCACCAGTTTGGGGACTCTCTCCCCGCGTTCTGTAAGAATCGGCGGCAGCCAGCAGGCGGGGAGGCGGGGGCACGTGTTTGGATGTGGGTGCTTGTGTAACCAGTTCCCCAAGCGCCAGCCCCGACAGCGCTCCTTCGGGAGGCTGGTCCGAGCCCCTGTTTCCGCCGCGGCGCAGGAAGGGTTGGGGTTCCGCTGCCTGCACCAGGCAAGAGCACCCCGAGCAAAGGAAGAAGACGACTTGCCTCCGGAGCTATCACTGGGGAGTGGGAATTTGGAAAGTTCCCCAACTAGGGACACACGTGACCTCCTTCGGAAAGTAGTTCCGACTGTGGCCCGTGTATCCTTCCACCTCCTTTTGAACCCTCCTAGGTCTCCTCGCCCCGCCCACTCGCTGGGCTGCAGCTTCCTACCGTTCCGTACTTTCCACTCAACCCGGTAACCCCAAACGTGCACGGTCCGGCCGGGGCGCGCGGAGCCTGGCCCCGGGCGATCCATCCTGCCGGGTTTTCACGGCGGCCAAGGGGGGGCGGGGCTAGGTGGTCTCTGAGAACCGAGCTTGACTCCGACGCCGCGAACCGACCTGGAGCCCGAGGGGAAAGATGCTCGACTCTCTTGGGGGCACCGGAGCGGGCGCAGGAGAGGCCTGCGGGGTGCGTCCCACTCACAGGGATCCTCTTTCAGTTCATTTAGATAGGTGCCCTTTGGGCCCTTGAAATTCAACGGCTATGTGTTCACGTTCAGCACGCTCGGCTGAGAGCTTTCATTTTTAGGGCAAACGAGCCGAGTTACCGGGGAAGCGAGAGGTGGGGCGCTGCAAGGGAGCCGGATGAGGTGATACACGCTGGCGACACAATAGCAGGTTGCTCTTTGTGCTAAGACTGACACCATGAGGACACAGATTTGGGGGAAGGGGGAATCTCTAGGCAAAGGCTGTTACAGTCAAATCTCTGCGAACGATTGTGATCCGACAGCGGTGCAAAAGGAAAGAGCGAATGCAGTCCACGCCGCGGAAATCTAGGGGTAGAGGCAAGGGGGGAGGGTATTCCCCTTGCAGGGACCGTCCCTGCATTTCCCTCTACACTGAGCAGCGTGGTCACCTGGTCCTTTTCACCTGTGCACAGGTAACCTCAGACTCGAGTCAGTGACACTGCTCAACGCACCCATCTCAGCTTTCATCATCAGTCCTCCACCCCCGCCCCACAACAGCCTACCCTGCCTCCGGCTGGGTTTCTGGGCAGAGGCCGAGGCTTAGCTCGTTATCCTCGCCTCGCGTTGCTGCAAAAGCCGCAGCAAGTGCAGCTGCAGGCTGGCGGCTGGGAACCGGCCCGAGCAAGCCCCAGGCAGCTACACTGGGCATGCTCAGTAGAGCCTGCGGCTTGGGGACTCTGCGCTCGCACCCAGAGCTACCGCTCTGCCCCCTCCTACCGCCCCCTGCCCTGCCCTGCCCTCCCCTCGCCCGGCGCGGTCCCGTCCGCCTCTCGCTCGCCTCCCGCCTCCCCTCGGTCTTCCGAGGCGCCCGGGCTCCCGGCGCGGCGGCGGAGGGGGCGGGCAGGCCGGCGGGCGGTGATGTGGCGGGACTCTTTATGCGCTGCGGCAGGATACGCGCTCGGCGCTGGGACGCGACTGCGCTCAGTTCTCTCCTCTCGGAAGCTGCAGCCATGATGGAAGTTTGAGAGTTGAGCCGCTGTGAGGCGAGGCCGGGCTCAGGCGAGGGAGATGAGAGACGGCGGCGGCCGCGGCCCGGAGCCCCTCTCAGCGCCTGTGAGCAGCCGCGGGGGCAGCGCCCTCGGGGAGCCGGCCGGCCTGCGGCGGCGGCAGCGGCGGCGTTTCTCGCCTCCTCTTCGTCTTTTCTAACCGTGCAGCCTCTTCCTCGGCTTCTCCTGAAAGGGAAGGTGGAAGCCGTGGGCTCGGGCGGGAGCCGGCTGAGGCGCGGCGGCGGCGGCGGCACCTCCCGCTCCTGGAGCGGGGGGGAGAAGCGGCGGCGGCGGCGGCCGCGGCGGCTGCAGCTCCAGGGAGGGGGTCTGAGTCGCCTGTCACCATTTCCAGGGCTGGGAACGCCGGAGAGTTGGTCTCTCCCCTTCTACTGCCTCCAACACGGCGGCGGCGGCGGCTGGCACATCCAGGGACCCGGGCCGGTTTTAAACCTCCCGTGCGCCGCCGCCGCACCCCCCGTGGCCCGGGCTCCGGAGGCCGCCGGCGGAGGCAGCCGTTCGGAGGATTATTCGTCTTCTCCCCATTCCGCTGCCGCCGCTGCCAGGCCTCTGGCTGCTGAGGAGAAGCAGGCCCAGTCGCTGCAACCATCCAGCAGCCGCCGCAGCAGCCATTACCCGGCTGCGGTCCAGAGCCAAGCGGCGGCAGAGCGAGGGGCATCAGCTACCGCCAAGTCCAGAGCCATTTCCATCCTGCAGAAGAAGCCCCGCCACCAGCAGCTTCTGCCATCTCTCTCCTCCTTTTTCTTCAGCCACAGGCTCCCAGACATGACAGCCATCATCAAAGAGATCGTTAGCAGAAACAAAAGGAGATATCAAGAGGATGGATTCGACTTAGACTTGACCTGTATCCATTTCTGCGGCTGCTCCTCTTTACCTTTCTGTCACTCTCTTAGAACGTGGGAGTAGACGGATGCGAAAATGTCCGTAGTTTGGGTGACTATAACATTTAACCCTGGTCAGGTTGCTAGGTCATATATTTTGTGTTTCCTTTCTGTGTATTCAACCTAGGGTGTGTTTGGCTAGACGGAACTCTTGCCTGGTTGCAAGTGTCAAGCCACCGATTGCTTTCTTAGGCTATCTATATGGTCTCTTCCTGAGGGCTATTGTCCGTTAATACAGAATACAGTACACTGTTAGTGGATTAGCGAGCTCGGTAATCCGGTCTCCTAAATGAACAAAAAAGTAGACGCTTTTTGAGGTTGAGCATATTTCGATTAAATCTTGGCTTAGGCCCTAGATCAAGGGTTTAGATCAGAATAAAATGAAAATTAGTGTTGCACGTACGCATATTGCATCAGAATCTTGCAGTGATTGTTTTAGTTTCCTGAGTTGCATTGATAGATTCTTTTAAAATATGACTGATTTGCATAACTTTAGAAGCAGAATCATTTTCAGTATATATGGTGCACATTGAGGGCAAAAAGTAGTTTTGTTAATGTTTAAACTTAAGTTACCTACAACTTTGAACTGTATGTAGAAGTTTTGTAGTTTGAAGTCAATAGTGCCATAATATACCTTATAAGGCGTTCTTACTAGATCTTTGTTATATTTACCTTTTTCTCTCCCTATGGGGTGATGTAGGATAGTGCTTGAAATTTGCACTTCAGTAGCATTTAATGTTCAGTGCTCTTGTCATAAACATAGAATGGATATTGAGTAGTTTCTGATCCCAGATGGTAATGTGTAGGTTCAAGGGTATTGTGTGTAGCAAGTGAAGATTGCAGAAATAAAACTTCAGTTCATGCTTGAAATTTAAGTATTGTTGTGATGCCAGAATTGCTGCTCACCGTTTTTAGGTTTCAGGTCCTCTGACACCTTTTGGTATCGTTAATTTTACTGATTTGTGTAGAATGTCAGTTGTATTTTACCAGCTAATATCTAGAAATGCTGGCAAGAGGGGTTTACTCCAGCTTTAGATTG
